In [1]:
import pandas as pd
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np
from utils.feature_selection import FeatureSelection
from utils.regression import Regression

In [2]:
# Attempt 1, straight time prediction

In [3]:
def generate_filtered_combined_data(data_isolated_agg, data_coscheduled_agg, data_inhib_coscheduled_agg, app_a_allowed: set, app_b_allowed: set) -> pd.DataFrame:
    # Filter data based on Num Nodes == 8 (as per original script logic)
    data_isolated_agg = data_isolated_agg[data_isolated_agg["Num Nodes"] == 8]
    data_coscheduled_agg = data_coscheduled_agg[data_coscheduled_agg["Num Nodes"] == 8]
    data_inhib_coscheduled_agg = data_inhib_coscheduled_agg[data_inhib_coscheduled_agg["Num Nodes"] == 8]

    # --- Step 1: Get all unique (App, Num Nodes) pairs from isolated data ---
    isolated_lookup = data_isolated_agg.set_index(['App', 'Num Nodes'])['MPI Time Average Mean']

    # --- Step 2: Get all unique inhib configs (per Num Nodes) ---
    inhib_config_cols = ['Inhib Message Size', 'Inhib Wait Time (us)', 'Inhib Comm Sparsity', 'Inhib Comm Mode']
    inhib_configs = data_inhib_coscheduled_agg[['Num Nodes'] + inhib_config_cols].drop_duplicates()

    # --- Step 3: Get all unique apps per Num Nodes from coscheduled data ---
    apps_per_nodes = (
        pd.concat([
            data_coscheduled_agg[['App A', 'Num Nodes']].rename(columns={'App A': 'App'}),
            data_coscheduled_agg[['App B', 'Num Nodes']].rename(columns={'App B': 'App'}),
        ])
        .drop_duplicates()
    )

    # --- Step 4: Get unique (App A, App B, Num Nodes) combos ---
    rows = []

    for num_nodes, group in apps_per_nodes.groupby('Num Nodes'):
        apps = group['App'].unique()
        for app_a, app_b in combinations(sorted(apps), 2):
            # --- Filter Logic: Only append rows if App A and App B are in their respective allowed sets ---
            if app_a not in app_a_allowed or app_b not in app_b_allowed:
                continue

            row = {}
            row['App A'] = app_a
            row['App B'] = app_b
            row['Num Nodes'] = num_nodes

            # --- Isolated times ---
            row['App A Isolated MPI Time'] = isolated_lookup.get((app_a, num_nodes), None)
            row['App B Isolated MPI Time'] = isolated_lookup.get((app_b, num_nodes), None)

            # --- Inhib co-scheduled times for App A and App B ---
            node_inhib_configs = inhib_configs[inhib_configs['Num Nodes'] == num_nodes]

            for _, cfg in node_inhib_configs.iterrows():
                cfg_filter = (
                    (data_inhib_coscheduled_agg['Num Nodes'] == num_nodes) &
                    (data_inhib_coscheduled_agg['Inhib Message Size'] == cfg['Inhib Message Size']) &
                    (data_inhib_coscheduled_agg['Inhib Wait Time (us)'] == cfg['Inhib Wait Time (us)']) &
                    (data_inhib_coscheduled_agg['Inhib Comm Sparsity'] == cfg['Inhib Comm Sparsity']) &
                    (data_inhib_coscheduled_agg['Inhib Comm Mode'] == cfg['Inhib Comm Mode'])
                )

                cfg_label = (
                    f"MsgSz={cfg['Inhib Message Size']}_"
                    f"Wait={cfg['Inhib Wait Time (us)']}us_"
                    f"Sparsity={cfg['Inhib Comm Sparsity']}_"
                    f"Mode={cfg['Inhib Comm Mode']}"
                )

                for app, col_prefix in [(app_a, 'App A'), (app_b, 'App B')]:
                    app_filter = cfg_filter & (data_inhib_coscheduled_agg['App'] == app)
                    match = data_inhib_coscheduled_agg[app_filter]
                    col_name = f"{col_prefix} Co-Scheduled MPI Time with Inhib [{cfg_label}]"
                    row[col_name] = match['MPI Time Average Mean'].values[0] if len(match) > 0 else None

            # --- Co-scheduled time with each other (from data_coscheduled_agg) ---
            cosched_filter = (
                (data_coscheduled_agg['Num Nodes'] == num_nodes) &
                (data_coscheduled_agg['App A'] == app_a) &
                (data_coscheduled_agg['App B'] == app_b)
            )
            cosched_match = data_coscheduled_agg[cosched_filter]

            # Also check reversed order
            cosched_filter_rev = (
                (data_coscheduled_agg['Num Nodes'] == num_nodes) &
                (data_coscheduled_agg['App A'] == app_b) &
                (data_coscheduled_agg['App B'] == app_a)
            )
            cosched_match_rev = data_coscheduled_agg[cosched_filter_rev]

            if len(cosched_match) > 0:
                row['App A Co-Scheduled MPI Time with App B'] = cosched_match['App A MPI Time Average Mean'].values[0]
                row['App B Co-Scheduled MPI Time with App A'] = cosched_match['App B MPI Time Average Mean'].values[0]
            elif len(cosched_match_rev) > 0:
                # A and B are swapped in the source df, so flip the columns
                row['App A Co-Scheduled MPI Time with App B'] = cosched_match_rev['App B MPI Time Average Mean'].values[0]
                row['App B Co-Scheduled MPI Time with App A'] = cosched_match_rev['App A MPI Time Average Mean'].values[0]
            else:
                row['App A Co-Scheduled MPI Time with App B'] = None
                row['App B Co-Scheduled MPI Time with App A'] = None

            rows.append(row)

    data_combined_training_df = pd.DataFrame(rows)
    return data_combined_training_df

def double_combined_dataset(data_df):

    # Double the dataset size by splitting App A and App B targets into two rows
    # Identify column categories
    app_a_isolated = 'App A Isolated MPI Time'
    app_b_isolated = 'App B Isolated MPI Time'

    inhib_a_cols = [c for c in data_df.columns if c.startswith('App A Co-Scheduled MPI Time with Inhib')]
    inhib_b_cols = [c for c in data_df.columns if c.startswith('App B Co-Scheduled MPI Time with Inhib')]

    cosched_a_col = 'App A Co-Scheduled MPI Time with App B'
    cosched_b_col = 'App B Co-Scheduled MPI Time with App A'

    # --- Row type 1: original row, drop "App B Co-Scheduled Time with App A" ---
    data_df_type1 = data_df.drop(columns=[cosched_b_col]).copy()

    # --- Row type 2: swap App A <-> App B, keep only "App A Co-Scheduled Time with App B" (post-swap) ---
    data_df_type2 = data_df.copy()

    # Swap App A / App B names
    data_df_type2['App A'] = data_df['App B']
    data_df_type2['App B'] = data_df['App A']

    # Swap isolated times
    data_df_type2[app_a_isolated] = data_df[app_b_isolated]
    data_df_type2[app_b_isolated] = data_df[app_a_isolated]

    # Swap inhibitor columns pairwise
    # inhib_a_cols[i] pairs with inhib_b_cols[i] (same param suffix)
    for col_a, col_b in zip(inhib_a_cols, inhib_b_cols):
        data_df_type2[col_a] = data_df[col_b]
        data_df_type2[col_b] = data_df[col_a]

    # After swap, "App A Co-Scheduled Time with App B" should hold what was cosched_b_col
    data_df_type2[cosched_a_col] = data_df[cosched_b_col]

    # Drop the cosched_b column (only keep cosched_a)
    data_df_type2 = data_df_type2.drop(columns=[cosched_b_col])

    # --- Combine ---
    data_df_extended = pd.concat([data_df_type1, data_df_type2], ignore_index=True)
    data_df_extended
    return data_df_extended

In [4]:
# Training and testing split of data

data_isolated_agg = pd.read_csv("../processed_data/data_isolated_agg.csv", index_col=0)
data_isolated_agg = data_isolated_agg[data_isolated_agg["Num Nodes"] == 8]
data_coscheduled_agg = pd.read_csv("../processed_data/data_coscheduled_agg.csv", index_col=0)
data_coscheduled_agg = data_coscheduled_agg[data_coscheduled_agg["Num Nodes"] == 8]
data_inhib_coscheduled_agg = pd.read_csv("../processed_data/data_inhib_coscheduled_agg.csv", index_col=0)
data_inhib_coscheduled_agg = data_inhib_coscheduled_agg[data_inhib_coscheduled_agg["Num Nodes"] == 8]

# Generate training dataframe
app_a_allowed = {"beatnik", "lammps", "minife", "fiesta", "minivite", "quicksilver", "minivite"}
app_b_allowed = {"beatnik", "lammps", "minife", "fiesta", "minivite", "quicksilver", "minivite"}
# app_a_allowed = set(data_isolated_agg["App"].unique())
# app_b_allowed = set(data_isolated_agg["App"].unique())
data_training_df = generate_filtered_combined_data(data_isolated_agg, data_coscheduled_agg, data_inhib_coscheduled_agg, app_a_allowed, app_b_allowed)
data_training_df = double_combined_dataset(data_training_df)

app_a_allowed = {"laghos", "kripke", "amg"}
# app_a_allowed = set(data_isolated_agg["App"].unique())
app_b_allowed = set(data_isolated_agg["App"].unique())
data_testing_df = generate_filtered_combined_data(data_isolated_agg, data_coscheduled_agg, data_inhib_coscheduled_agg, app_a_allowed, app_b_allowed)
data_testing_df = double_combined_dataset(data_testing_df)


# All data
data_all_df = generate_filtered_combined_data(data_isolated_agg, data_coscheduled_agg, data_inhib_coscheduled_agg, set(data_isolated_agg["App"].unique()), set(data_isolated_agg["App"].unique()))
data_all_df = double_combined_dataset(data_all_df)

In [18]:
# Feature selection and training
fs = FeatureSelection()
# all_selections = fs.get_feature_matrix(data_training_df, k=3)
all_selections = fs.get_feature_matrix(data_all_df, k=10)
consensus_df = fs.get_consensus()
app_time_lookup = data_isolated_agg.set_index('App')['MPI Time Average Mean']

results_matrix_squared = {}
predictions_matrix_squared_training = {}
predictions_matrix_squared_testing = {}
reg_squared = {}
for i in all_selections.keys():
    reg_squared[i] = Regression(loss_type="squared", scaler_type="minmax")
    reg_squared[i].train(data_training_df, all_selections[i], loss_type="squared")
    predictions_matrix_squared_training[i] = reg_squared[i].predict(data_training_df, all_selections[i])
    predictions_matrix_squared_testing[i] = reg_squared[i].predict(data_testing_df, all_selections[i])
    predictions_matrix_squared_training[i]['App A Isolated MPI Time'] = predictions_matrix_squared_training[i]['App A'].map(app_time_lookup)
    predictions_matrix_squared_training[i]['App B Isolated MPI Time'] = predictions_matrix_squared_training[i]['App B'].map(app_time_lookup)
    predictions_matrix_squared_testing[i]['App A Isolated MPI Time'] = predictions_matrix_squared_testing[i]['App A'].map(app_time_lookup)
    predictions_matrix_squared_testing[i]['App B Isolated MPI Time'] = predictions_matrix_squared_testing[i]['App B'].map(app_time_lookup)
    
""" results_matrix_huber = {}
predictions_matrix_huber_training = {}
predictions_matrix_huber_testing = {}
reg_huber = {}
for i in all_selections.keys():
    reg_huber[i] = Regression(loss_type="huber")
    reg_huber[i].train(data_training_df, all_selections[i], loss_type="huber")
    predictions_matrix_huber_training[i] = reg_huber[i].predict(data_training_df, all_selections[i])
    predictions_matrix_huber_testing[i] = reg_huber[i].predict(data_testing_df, all_selections[i])
    predictions_matrix_huber_training[i]['App A Isolated MPI Time'] = predictions_matrix_huber_training[i]['App A'].map(app_time_lookup)
    predictions_matrix_huber_training[i]['App B Isolated MPI Time'] = predictions_matrix_huber_training[i]['App B'].map(app_time_lookup)
    predictions_matrix_huber_testing[i]['App A Isolated MPI Time'] = predictions_matrix_huber_testing[i]['App A'].map(app_time_lookup)
    predictions_matrix_huber_testing[i]['App B Isolated MPI Time'] = predictions_matrix_huber_testing[i]['App B'].map(app_time_lookup)

results_matrix_mae = {}
predictions_matrix_mae_training = {}
predictions_matrix_mae_testing = {}
reg_mae = {}
for i in all_selections.keys():
    reg_mae[i] = Regression(loss_type="mae")
    reg_mae[i].train(data_training_df, all_selections[i], loss_type="mae")
    predictions_matrix_mae_training[i] = reg_mae[i].predict(data_training_df, all_selections[i])
    predictions_matrix_mae_testing[i] = reg_mae[i].predict(data_testing_df, all_selections[i])
    predictions_matrix_mae_training[i]['App A Isolated MPI Time'] = predictions_matrix_mae_training[i]['App A'].map(app_time_lookup)
    predictions_matrix_mae_training[i]['App B Isolated MPI Time'] = predictions_matrix_mae_training[i]['App B'].map(app_time_lookup)
    predictions_matrix_mae_testing[i]['App A Isolated MPI Time'] = predictions_matrix_mae_testing[i]['App A'].map(app_time_lookup)
    predictions_matrix_mae_testing[i]['App B Isolated MPI Time'] = predictions_matrix_mae_testing[i]['App B'].map(app_time_lookup)

results_matrix_log_cosh = {}
predictions_matrix_log_cosh_training = {}
predictions_matrix_log_cosh_testing = {}
reg_log_cosh = {}
for i in all_selections.keys():
    reg_log_cosh[i] = Regression(loss_type="log_cosh")
    reg_log_cosh[i].train(data_training_df, all_selections[i], loss_type="log_cosh")
    predictions_matrix_log_cosh_training[i] = reg_log_cosh[i].predict(data_training_df, all_selections[i])
    predictions_matrix_log_cosh_testing[i] = reg_log_cosh[i].predict(data_testing_df, all_selections[i])
    predictions_matrix_log_cosh_training[i]['App A Isolated MPI Time'] = predictions_matrix_log_cosh_training[i]['App A'].map(app_time_lookup)
    predictions_matrix_log_cosh_training[i]['App B Isolated MPI Time'] = predictions_matrix_log_cosh_training[i]['App B'].map(app_time_lookup)
    predictions_matrix_log_cosh_testing[i]['App A Isolated MPI Time'] = predictions_matrix_log_cosh_testing[i]['App A'].map(app_time_lookup)
    predictions_matrix_log_cosh_testing[i]['App B Isolated MPI Time'] = predictions_matrix_log_cosh_testing[i]['App B'].map(app_time_lookup) """

Using dataframe: 90 rows, 486 columns
  Preserved columns : ['App A', 'App B']
  Target column     : App A Co-Scheduled MPI Time with App B
  Feature columns   : 483
  (Always including 'App A Isolated MPI Time' separately)

Standardization complete (StandardScaler).
  X shape: (90, 482), y shape: (90,)

[1] Pearson Correlation ...
    Top 10: ['App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=100000us_Sparsity=0.7_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=1000000us_Sparsity=0.7_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=100000us_Sparsity=0.3_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=10000us_Sparsity=0.5_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=1000000us_Sparsity=0.7_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=100000us_Sparsity=0.7_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=10000us_Sparsity=1.0_Mode=d]', 'App A Co-Scheduled MPI Time w

/home/akhil/hpcResearch/python_venvs/ml_analysis/lib64/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.418e+03, tolerance: 3.765e+01
  model = cd_fast.enet_coordinate_descent(
/home/akhil/hpcResearch/python_venvs/ml_analysis/lib64/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.382e+03, tolerance: 3.765e+01
  model = cd_fast.enet_coordinate_descent(
/home/akhil/hpcResearch/python_venvs/ml_analysis/lib64/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter

    Best alpha: 0.166088  (non-zero features ≈ 10)
    Top 10: ['App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=100000us_Sparsity=0.7_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=1000us_Sparsity=0.7_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=100us_Sparsity=0.7_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=10000us_Sparsity=0.3_Mode=d]', 'App B Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.5_Mode=d]', 'App B Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=1.0_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=2000000_Wait=10000us_Sparsity=0.7_Mode=d]', 'App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=1000000us_Sparsity=0.7_Mode=d]', 'App B Co-Scheduled MPI Time with Inhib [MsgSz=40000_Wait=100us_Sparsity=0.7_Mode=d]', 'App B Co-Scheduled MPI Time with Inhib [MsgSz=20000_Wait=1000us_Sparsity=0.3_Mode=d]']

[5] Recursive Feature Elimination (RFE with

' results_matrix_huber = {}\npredictions_matrix_huber_training = {}\npredictions_matrix_huber_testing = {}\nreg_huber = {}\nfor i in all_selections.keys():\n    reg_huber[i] = Regression(loss_type="huber")\n    reg_huber[i].train(data_training_df, all_selections[i], loss_type="huber")\n    predictions_matrix_huber_training[i] = reg_huber[i].predict(data_training_df, all_selections[i])\n    predictions_matrix_huber_testing[i] = reg_huber[i].predict(data_testing_df, all_selections[i])\n    predictions_matrix_huber_training[i][\'App A Isolated MPI Time\'] = predictions_matrix_huber_training[i][\'App A\'].map(app_time_lookup)\n    predictions_matrix_huber_training[i][\'App B Isolated MPI Time\'] = predictions_matrix_huber_training[i][\'App B\'].map(app_time_lookup)\n    predictions_matrix_huber_testing[i][\'App A Isolated MPI Time\'] = predictions_matrix_huber_testing[i][\'App A\'].map(app_time_lookup)\n    predictions_matrix_huber_testing[i][\'App B Isolated MPI Time\'] = predictions_matr

In [19]:
import os

def plot_row_metrics_separate(df, output_folder='row_plots'):
    """
    Creates a separate barplot for each row in the dataframe.
    """
    
    # 1. Identify columns to plot based on instructions
    # We explicitly define the fixed columns and filter for y_pred columns
    fixed_cols = ['App A Isolated MPI Time', 'y_true']
    
    # Filter columns that look like predictions
    pred_cols = [col for col in df.columns if 'y_pred' in col]
    
    # Combine for the plotting list
    plot_cols = fixed_cols + pred_cols
    
    # Ensure all expected columns exist in the dataframe to avoid crashes
    missing = [col for col in plot_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Columns not found in dataframe: {missing}")

    # 2. Define the mapping for labels
    # Fixed mappings
    col_labels_map = {
        'App A Isolated MPI Time': 'Isolated Time',
        'y_true': 'Coscheduled Actual time'
    }
    
    # Create the label list corresponding to plot_cols
    x_labels = []
    for col in plot_cols:
        if col in col_labels_map:
            x_labels.append(col_labels_map[col])
        elif 'y_pred' in col:
            # Split by '_' and take the last element (e.g., 'Ridge', 'KNN (k=3)')
            x_labels.append(col.split('_')[-1])
        else:
            x_labels.append(col) # Fallback

    # 3. Setup output folder
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # 4. Iterate through rows
    print(f"Starting plotting {len(df)} rows...")
    for idx, row in df.iterrows():
        # Create Figure
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Extract values for the current row
        # Convert to float to ensure numeric plotting
        y_values = row[plot_cols].astype(float)
        
        # Plot Bar
        bars = ax.bar(x_labels, y_values, color='skyblue', edgecolor='black')
        
        # Set Titles and Labels
        ax.set_ylabel('Time (in seconds)')
        ax.set_xlabel('Method')
        ax.set_title(f'Row {idx}: App Performance Metrics')
        
        # Rotate x-labels slightly for readability if they are long
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        
        # Add value labels on top of bars (optional, for clarity)
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}',
                    ha='center', va='bottom', fontsize=9)
        
        # Tight layout to prevent label cut-off
        plt.tight_layout()
        
        # Save plot
        # filename = os.path.join(output_folder, f'plot_row_{idx}.png')
        # fig.savefig(filename)
        # plt.close(fig) # Close figure to free memory

    print(f"Finished. Plots saved to '{output_folder}' folder.")

# --- Example Usage ---
# Assuming your dataframe is already named 'df'

# Uncomment the lines below to run this on your actual dataframe:
# plot_row_metrics_separate(predictions_matrix_squared_testing["Correlation"], output_folder='my_app_plots')


In [20]:
reg_squared["Correlation"].get_error_metrics(predictions_matrix_squared_training["Correlation"])

,model,mse,mae,r2,rmse,mape,wmse,wmae,wrmse,wmape
0,App A Isolated MPI Time,64.5149,5.3705,0.9691,8.0321,8.1901,69.7459,5.6828,8.3514,8.6826
1,Ridge,13.5599,2.8608,0.9935,3.6824,7.2428,13.9936,2.8841,3.7408,7.3081
2,ElasticNet,15.6420,3.2035,0.9925,3.9550,8.5474,15.9276,3.2028,3.9909,8.5339
3,SVR (RBF),51.5166,4.9935,0.9753,7.1775,10.5138,49.8358,4.8845,7.0594,10.3982
4,KNN (k=3),13.3132,2.6253,0.9936,3.6487,6.6186,13.6187,2.6606,3.6903,6.7340
5,Gradient Boosting,11.8187,2.4949,0.9943,3.4378,6.0117,12.4176,2.5461,3.5239,6.1637
6,Random Forest,11.6924,2.4656,0.9944,3.4194,5.7967,12.3072,2.5210,3.5082,5.9724


In [21]:
reg_squared["Correlation"].get_error_metrics(predictions_matrix_squared_testing["Correlation"])

,model,mse,mae,r2,rmse,mape,wmse,wmae,wrmse,wmape
0,App A Isolated MPI Time,167.7448,5.8738,0.9524,12.9516,10.1948,282.5550,7.8857,16.8094,12.6853
1,Ridge,129.6962,6.6441,0.9632,11.3884,15.7917,175.0353,7.6555,13.2301,16.6515
2,ElasticNet,127.3999,6.8582,0.9639,11.2872,18.0166,172.7066,7.8304,13.1418,18.6297
3,SVR (RBF),1719.0472,17.8812,0.5123,41.4614,24.4831,1651.3049,17.9685,40.6363,24.3714
4,KNN (k=3),606.5933,11.0784,0.8279,24.6291,14.8327,644.7333,12.1373,25.3916,16.1479
5,Gradient Boosting,619.5862,10.9212,0.8242,24.8915,14.3273,668.6090,12.1321,25.8575,15.7952
6,Random Forest,615.8494,10.9185,0.8253,24.8163,14.5202,666.0599,12.1364,25.8081,15.9576


In [22]:
for i in all_selections.keys():
    print(f"Method: {i}")
    for j in all_selections[i]:
        print(j)

Method: Correlation
App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=100000us_Sparsity=0.7_Mode=d]
App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=1000000us_Sparsity=0.7_Mode=d]
App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=100000us_Sparsity=0.3_Mode=d]
App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=10000us_Sparsity=0.5_Mode=d]
App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=1000000us_Sparsity=0.7_Mode=d]
App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=100000us_Sparsity=0.7_Mode=d]
App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=10000us_Sparsity=1.0_Mode=d]
App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=10000us_Sparsity=0.3_Mode=d]
App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=10000us_Sparsity=0.5_Mode=d]
App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=1000us_Sparsity=0.7_Mode=d]
App A Isolated MPI Time
Method: MutualInfo
App A Co-Scheduled MPI Time with Inhib [MsgSz=4000_Wait=100000us_Sparsity=0.5_Mod

In [11]:
data_all_df

,App A,App B,Num Nodes,App A Isolated MPI Time,App B Isolated MPI Time,App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.1_Mode=d],App B Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.1_Mode=d],App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.3_Mode=d],App B Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.3_Mode=d],App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.5_Mode=d],...,App B Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.1_Mode=d],App A Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.3_Mode=d],App B Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.3_Mode=d],App A Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.5_Mode=d],App B Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.5_Mode=d],App A Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.7_Mode=d],App B Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.7_Mode=d],App A Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=1.0_Mode=d],App B Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=1.0_Mode=d],App A Co-Scheduled MPI Time with App B
0,amg,beatnik,8,12.778086,99.906362,27.880246,126.700893,14.685140,117.027344,13.073605,...,190.251674,35.622991,211.793527,39.671094,188.018973,40.329576,199.823661,41.997991,347.267857,19.768080
1,amg,fiesta,8,12.778086,65.675446,27.880246,84.057533,14.685140,55.426004,13.073605,...,68.821931,35.622991,59.038225,39.671094,64.025000,40.329576,64.335547,41.997991,95.488170,13.131256
2,amg,kripke,8,12.778086,17.006585,27.880246,20.994085,14.685140,17.226116,13.073605,...,19.548661,35.622991,20.789788,39.671094,22.184877,40.329576,22.638839,41.997991,21.505022,14.017204
3,amg,laghos,8,12.778086,54.944978,27.880246,184.186384,14.685140,114.130692,13.073605,...,160.097656,35.622991,215.852121,39.671094,243.406808,40.329576,238.407366,41.997991,272.857143,14.706105
4,amg,lammps,8,12.778086,17.381222,27.880246,20.600067,14.685140,16.306914,13.073605,...,23.874604,35.622991,28.386836,39.671094,26.668432,40.329576,30.012612,41.997991,29.754408,13.311166
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,quicksilver,minife,8,133.995497,13.601356,134.511468,20.535279,135.182919,14.775982,137.223538,...,30.617578,136.910006,35.439453,133.390575,36.365513,134.988834,41.655469,133.923733,43.692857,134.894699
86,tricount,minife,8,204.072545,13.601356,261.073661,20.535279,231.395647,14.775982,227.660714,...,30.617578,392.374442,35.439453,443.263393,36.365513,410.602679,41.655469,444.787388,43.692857,221.913504
87,quicksilver,minivite,8,133.995497,51.597266,134.511468,64.183092,135.182919,61.065904,137.223538,...,88.622824,136.910006,95.423326,133.390575,89.561942,134.988834,103.508036,133.923733,126.427902,136.762433
88,tricount,minivite,8,204.072545,51.597266,261.073661,64.183092,231.395647,61.065904,227.660714,...,88.622824,392.374442,95.423326,443.263393,89.561942,410.602679,103.508036,444.787388,126.427902,225.238839


In [ ]:
# Method 2 - slowdown prediction

In [6]:
data_all_df.columns

Index(['App A', 'App B', 'Num Nodes', 'App A Isolated MPI Time',
       'App B Isolated MPI Time',
       'App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.1_Mode=d]',
       'App B Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.1_Mode=d]',
       'App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.3_Mode=d]',
       'App B Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.3_Mode=d]',
       'App A Co-Scheduled MPI Time with Inhib [MsgSz=2000_Wait=0us_Sparsity=0.5_Mode=d]',
       ...
       'App B Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.1_Mode=d]',
       'App A Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.3_Mode=d]',
       'App B Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.3_Mode=d]',
       'App A Co-Scheduled MPI Time with Inhib [MsgSz=4000000_Wait=1000000us_Sparsity=0.5_Mode=d]',
       'App B Co-Scheduled MPI Time

In [7]:
def convert_to_slowdown(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converts Co-Scheduled MPI Time columns into Slowdown metrics.
    
    Logic:
    1. Keeps all columns except those containing 'Co-Scheduled'.
    2. For columns containing 'Co-Scheduled':
       - Divides by the corresponding 'App A Isolated MPI Time' or 'App B Isolated MPI Time'.
       - Caps the result at 1.0 (slowdown cannot be less than 1.0).
       - Renames the column to include 'Slowdown' instead of 'MPI Time'.
       
    Args:
        df (pd.DataFrame): Input dataframe containing the performance metrics.
        
    Returns:
        pd.DataFrame: New dataframe with updated columns.
    """
    # Create a copy to avoid modifying the original dataframe
    new_df = df.copy()

    # Define the names of the isolated time columns (baselines)
    app_a_iso_col = 'App A Isolated MPI Time'
    app_b_iso_col = 'App B Isolated MPI Time'

    # Identify columns that need conversion
    # We look for columns containing the specific substring "Co-Scheduled"
    cols_to_convert = [col for col in df.columns if 'Co-Scheduled' in col]

    # Identify columns to keep as-is (everything else)
    # This implicitly keeps 'App A', 'App B', 'Num Nodes', and the 'Isolated' columns
    cols_to_keep = [col for col in df.columns if 'Co-Scheduled' not in col]

    # Start the new dataframe with the columns that remain unchanged
    final_df = new_df[cols_to_keep].copy()

    # Process each column that needs conversion
    for col in cols_to_convert:
        # Determine which App this column belongs to based on the column name prefix
        if col.startswith('App A'):
            baseline_col = app_a_iso_col
            # Generate new name: replace "Co-Scheduled MPI Time" with "Co-scheduled Slowdown"
            new_col_name = col.replace('Co-Scheduled MPI Time', 'Co-scheduled Slowdown')
        elif col.startswith('App B'):
            baseline_col = app_b_iso_col
            new_col_name = col.replace('Co-Scheduled MPI Time', 'Co-scheduled Slowdown')
        else:
            # Fallback if column format is unexpected, skip this column
            continue

        # Check if the baseline column exists in the dataframe
        if baseline_col not in final_df.columns:
            raise ValueError(f"Baseline column '{baseline_col}' not found in dataframe.")

        # Calculate Slowdown = Co-Scheduled Time / Isolated Time
        # Use division safely to handle potential division by zero (resulting in inf)
        # We assume valid benchmark data where isolated time > 0.
        slow_down = df[col] / final_df[baseline_col]
        
        # Replace infinities (caused by dividing by zero) with 1.0 (no slowdown)
        slow_down = slow_down.replace([np.inf, -np.inf], 1.0)
        
        # Apply the constraint: If slowdown < 1.0, cap at 1.0
        slow_down = slow_down.clip(lower=1.0)
        
        # Add the new column to the final dataframe
        final_df[new_col_name] = slow_down

    return final_df


In [ ]:
# Slowdown conversions
data_all_slowdown_df = convert_to_slowdown(data_all_df)
data_training_slowdown_df = convert_to_slowdown(data_training_df)
data_testing_slowdown_df = convert_to_slowdown(data_testing_df)

/tmp/ipykernel_128428/3360747714.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[new_col_name] = slow_down


In [ ]:
# Feature selection and training
fs = FeatureSelection()
# all_selections = fs.get_feature_matrix(data_training_df, k=3)
all_selections = fs.get_feature_matrix(data_all_df, k=10)
consensus_df = fs.get_consensus()
app_time_lookup = data_isolated_agg.set_index('App')['MPI Time Average Mean']

results_matrix_squared = {}
predictions_matrix_squared_training = {}
predictions_matrix_squared_testing = {}
reg_squared = {}
for i in all_selections.keys():
    reg_squared[i] = Regression(loss_type="squared", scaler_type="minmax")
    reg_squared[i].train(data_training_df, all_selections[i], loss_type="squared")
    predictions_matrix_squared_training[i] = reg_squared[i].predict(data_training_df, all_selections[i])
    predictions_matrix_squared_testing[i] = reg_squared[i].predict(data_testing_df, all_selections[i])
    predictions_matrix_squared_training[i]['App A Isolated MPI Time'] = predictions_matrix_squared_training[i]['App A'].map(app_time_lookup)
    predictions_matrix_squared_training[i]['App B Isolated MPI Time'] = predictions_matrix_squared_training[i]['App B'].map(app_time_lookup)
    predictions_matrix_squared_testing[i]['App A Isolated MPI Time'] = predictions_matrix_squared_testing[i]['App A'].map(app_time_lookup)
    predictions_matrix_squared_testing[i]['App B Isolated MPI Time'] = predictions_matrix_squared_testing[i]['App B'].map(app_time_lookup)